# Ноутбук 5 — оценка ПОСЛЕ дообучения и сравнение до/после

Гружу базовую модель + мой LoRA-адаптер (из checkpoint на Drive) и прогоняю те же бенчмарки
GQA-ru и MMBench-ru с тем же N и seed, что в ноутбуке 3. Сравниваю с базлайном.

Базлайн (до обучения): GQA-ru 0.466, MMBench-ru 0.642 (n=1000, seed=42).

## Библиотеки

Тот же набор версий, что и раньше, + peft (для адаптера). После установки перезапусти сессию.

In [ ]:
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" "huggingface_hub==0.24.6" "accelerate==0.33.0" "datasets==2.21.0" "peft==0.12.0" "pillow<12"

После перезапуска — проверка версий.

In [ ]:
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__)
print('GPU:', torch.cuda.is_available())

## Логин в HuggingFace

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Подключаю Google Drive (там лежит адаптер)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

ADAPTER_PATH = '/content/drive/MyDrive/vlm_project/checkpoints/checkpoint-100'
print('адаптер:', ADAPTER_PATH)
print('файлы:', os.listdir(ADAPTER_PATH))

## Гружу базовую модель + адаптер

Базу — в fp16, сверху навешиваю LoRA-адаптер и вплавляю его в веса (merge) для быстрого инференса.

In [ ]:
import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, AutoTokenizer
from peft import PeftModel

model_id = 'deepvk/llava-gemma-2b-lora'
base = LlavaForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
model = model.merge_and_unload()
processor = AutoProcessor.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)
print('модель с адаптером готова')

## Функции оценки

Ровно те же, что в ноутбуке 3 — иначе сравнение будет нечестным.

In [ ]:
import re

def ask(img, question, max_new=10):
    messages = [{'role': 'user', 'content': '<image>\n' + question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(images=[img], text=text, return_tensors='pt').to(model.device)
    inputs['pixel_values'] = inputs['pixel_values'].to(torch.float16)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


def normalize(s):
    s = str(s).lower().strip()
    s = re.sub(r'[^0-9a-zа-яё ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def gqa_correct(pred, gold):
    p, g = normalize(pred), normalize(gold)
    if not g:
        return False
    if p == g:
        return True
    if g in p.split():
        return True
    if p.startswith(g + ' ') or p.endswith(' ' + g):
        return True
    return False


def build_mmbench_options(ex):
    opts = {}
    for k in ['A', 'B', 'C', 'D']:
        v = ex.get(k)
        if v is not None and str(v).strip().lower() != 'nan' and str(v).strip() != '':
            opts[k] = v
    return opts


def parse_mmbench_letter(pred, options):
    m = re.search(r'[ABCD]', str(pred).upper())
    if m and m.group(0) in options:
        return m.group(0)
    pn = normalize(pred)
    for letter, txt in options.items():
        tn = normalize(txt)
        if tn and (pn == tn or tn in pn):
            return letter
    return None

## GQA-ru — оценка после обучения

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

gqa_q = load_dataset('deepvk/GQA-ru', 'testdev_balanced_instructions', split='testdev')
gqa_img = load_dataset('deepvk/GQA-ru', 'testdev_balanced_images', split='testdev')
id2img = {r['id']: r['image'] for r in gqa_img}

def eval_gqa(n):
    correct = 0
    total = 0
    for ex in tqdm(gqa_q.shuffle(seed=42).select(range(n)), total=n):
        img = id2img.get(ex['imageId'])
        if img is None:
            continue
        pred = ask(img, ex['question'] + '\nОтветь одним словом.', max_new=10)
        correct += int(gqa_correct(pred, ex['answer']))
        total += 1
    acc = correct / total if total else 0.0
    print(f'GQA accuracy после обучения на {total}: {acc:.3f}')
    return acc, total

gqa_acc, gqa_total = eval_gqa(1000)

## MMBench-ru — оценка после обучения

In [ ]:
mmb = load_dataset('deepvk/MMBench-ru', split='dev')

def eval_mmbench(n):
    correct = 0
    total = 0
    for ex in tqdm(mmb.shuffle(seed=42).select(range(n)), total=n):
        opts = build_mmbench_options(ex)
        if not opts:
            continue
        q = ex['question']
        if ex.get('hint') and str(ex['hint']).strip().lower() != 'nan':
            q = str(ex['hint']) + '\n' + q
        opts_text = '\n'.join(f'{k}. {v}' for k, v in opts.items())
        prompt = q + '\n' + opts_text + '\nОтветь только буквой правильного варианта.'
        pred = ask(ex['image'], prompt, max_new=5)
        letter = parse_mmbench_letter(pred, opts)
        correct += int(letter == ex['answer'])
        total += 1
    acc = correct / total if total else 0.0
    print(f'MMBench accuracy после обучения на {total}: {acc:.3f}')
    return acc, total

mmb_acc, mmb_total = eval_mmbench(1000)

## Сравнение до / после

In [ ]:
base_gqa = 0.466
base_mmb = 0.642
print('               до     ->  после     дельта')
print(f'GQA-ru      {base_gqa:.3f}  ->  {gqa_acc:.3f}    {gqa_acc - base_gqa:+.3f}')
print(f'MMBench-ru  {base_mmb:.3f}  ->  {mmb_acc:.3f}    {mmb_acc - base_mmb:+.3f}')

import json
res = {
    'gqa': {'before': base_gqa, 'after': round(gqa_acc, 4), 'n': gqa_total},
    'mmbench': {'before': base_mmb, 'after': round(mmb_acc, 4), 'n': mmb_total},
}
with open('after_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(res, f, ensure_ascii=False, indent=2)
print('\nсохранил after_metrics.json')

## Итог

Таблица до/после — главный результат проекта. Числа кладу в results, строю график,
дописываю выводы в README. Если прирост около нуля — это тоже честный результат
(базовую модель уже обучали на этих данных), опишем это в выводах.